In [1]:
# import libraries
import pandas as pd
import random as rnd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from bs4 import BeautifulSoup
import requests
import logging
from io import StringIO

%matplotlib inline

# setting up pseudo-random numbers for python standard and numpy random module.
rng = np.random.default_rng(seed=1234)

### Connecting the WebAPI and get the data 

#### Requests to fetch US Public Holidays from API
https://api.api-ninjas.com/v1/publicholidays?country=US

In [2]:
# Define API endpoint and parameters
HOLIDAY_API_URL = "https://api.api-ninjas.com/v1/publicholidays"
COUNTRY = "US"

# API key, Key is masked here
API_KEY = "XXXXXXXXXXXXXXXXXXXXXX"

try:
    # request to get public holiday data.
    response_holiday = requests.request("GET", f"{HOLIDAY_API_URL}?country={COUNTRY}", headers={"X-Api-Key": API_KEY})
except requests.exceptions.HTTPError as e:
    logging.error(f"HTTP Error occurred: {e}")
    raise SystemExit()
except requests.exceptions.ConnectionError as e:
    logging.error(f"Connection error occurred: {e}")
    raise SystemExit()
else:
    resp_holiday_data = response_holiday.json() # load json response into python object.

print("Lenght of the public poliday response: ", len(resp_holiday_data) )

Lenght of the public poliday response:  13


#### Requests to fetch AIR Quality of our selected cities
https://api-ninjas.com/api/airquality

In [3]:
# Define API endpoint and parameters
AQI_API_URL = "https://api.api-ninjas.com/v1/airquality"

CITIES = ["New York","Miami","Chicago","Los Angeles","Santa Monica","San Francisco"]

# API key
API_KEY = "XXXXXXXXXXXXXXXXXXXXXX"

# Dictionary to store response data for each city.
resp_city_data = {}

# loop to run API request for each city and store response in a dictionary
for ct in CITIES:
    try:
        # request to get public holiday data.
        response_ct = requests.request("GET", f"{AQI_API_URL}?city={ct}", headers={"X-Api-Key": API_KEY})
    except requests.exceptions.HTTPError as e:
        logging.error(f"HTTP Error occurred: {e}")
        raise SystemExit()
    except requests.exceptions.ConnectionError as e:
        logging.error(f"Connection error occurred: {e}")
        raise SystemExit()
    else:
        resp_city_data[ct] = response_ct.json() # load json response into python object.

print("Lenght of the AQI city response: ",len(resp_city_data))

Lenght of the AQI city response:  6


#### Get the Public Holiday and Air Quality Index data into the dataframe

In [4]:
print(resp_holiday_data[0]) # print the first element.

{'name': "New Year's Day", 'local_name': "New Year's Day", 'date': '2025-01-01', 'country': 'US', 'year': 2025, 'regions': [], 'federal': True}


In [5]:
# Public holiday data into dataframe
df_holidays = pd.DataFrame(resp_holiday_data)
df_holidays.head()

,name,local_name,date,country,year,regions,federal
0,New Year's Day,New Year's Day,2025-01-01,US,2025,[],True
1,"Martin Luther King, Jr. Day","Martin Luther King, Jr. Day",2025-01-20,US,2025,[],True
2,Presidents Day,Washington's Birthday,2025-02-17,US,2025,[],True
3,Good Friday,Good Friday,2025-04-18,US,2025,"[US-CT, US-DE, US-HI, US-IN, US-KY, US-LA, US-...",False
4,Memorial Day,Memorial Day,2025-05-26,US,2025,[],True


In [6]:
print(resp_city_data["New York"]) # print the first element.

{'CO': {'concentration': 370.18, 'aqi': 4}, 'NO2': {'concentration': 24.91, 'aqi': 31}, 'O3': {'concentration': 32.75, 'aqi': 27}, 'SO2': {'concentration': 2.35, 'aqi': 3}, 'PM2.5': {'concentration': 16.86, 'aqi': 52}, 'PM10': {'concentration': 19.2, 'aqi': 17}, 'overall_aqi': 52}


In [7]:
# AQI data into dataframe
df_city = pd.DataFrame(resp_city_data).T
df_city

,CO,NO2,O3,SO2,PM2.5,PM10,overall_aqi
New York,"{'concentration': 370.18, 'aqi': 4}","{'concentration': 24.91, 'aqi': 31}","{'concentration': 32.75, 'aqi': 27}","{'concentration': 2.35, 'aqi': 3}","{'concentration': 16.86, 'aqi': 52}","{'concentration': 19.2, 'aqi': 17}",52
Miami,"{'concentration': 100.09, 'aqi': 1}","{'concentration': 0.27, 'aqi': 0}","{'concentration': 92.65, 'aqi': 144}","{'concentration': 0.73, 'aqi': 1}","{'concentration': 2.78, 'aqi': 9}","{'concentration': 7.34, 'aqi': 6}",144
Chicago,"{'concentration': 217.28, 'aqi': 2}","{'concentration': 10.59, 'aqi': 13}","{'concentration': 50.63, 'aqi': 42}","{'concentration': 3.1, 'aqi': 4}","{'concentration': 8.68, 'aqi': 28}","{'concentration': 10.64, 'aqi': 9}",42
Los Angeles,"{'concentration': 142.68, 'aqi': 1}","{'concentration': 6.76, 'aqi': 8}","{'concentration': 88.73, 'aqi': 134}","{'concentration': 2.22, 'aqi': 3}","{'concentration': 2.9, 'aqi': 9}","{'concentration': 8.8, 'aqi': 8}",134
Santa Monica,"{'concentration': 137.55, 'aqi': 1}","{'concentration': 5.12, 'aqi': 6}","{'concentration': 90.18, 'aqi': 137}","{'concentration': 1.77, 'aqi': 2}","{'concentration': 2.8, 'aqi': 9}","{'concentration': 9.04, 'aqi': 8}",137
San Francisco,"{'concentration': 119.84, 'aqi': 1}","{'concentration': 1.28, 'aqi': 1}","{'concentration': 62.35, 'aqi': 60}","{'concentration': 1, 'aqi': 1}","{'concentration': 5.12, 'aqi': 16}","{'concentration': 12.16, 'aqi': 11}",60


#### Provding the meaningful header names

In [8]:
# Column headers for public holiday dataframe
holiday_cols = ["Holiday_Name", "Local_Name", "Date", "Country", "Year", "US_Regions", "Is_Federal"]
df_holidays.columns = holiday_cols
df_holidays.head()

,Holiday_Name,Local_Name,Date,Country,Year,US_Regions,Is_Federal
0,New Year's Day,New Year's Day,2025-01-01,US,2025,[],True
1,"Martin Luther King, Jr. Day","Martin Luther King, Jr. Day",2025-01-20,US,2025,[],True
2,Presidents Day,Washington's Birthday,2025-02-17,US,2025,[],True
3,Good Friday,Good Friday,2025-04-18,US,2025,"[US-CT, US-DE, US-HI, US-IN, US-KY, US-LA, US-...",False
4,Memorial Day,Memorial Day,2025-05-26,US,2025,[],True


In [9]:
# Make the city index as part of columns
df_city.reset_index(inplace=True)
df_city.head()

,index,CO,NO2,O3,SO2,PM2.5,PM10,overall_aqi
0,New York,"{'concentration': 370.18, 'aqi': 4}","{'concentration': 24.91, 'aqi': 31}","{'concentration': 32.75, 'aqi': 27}","{'concentration': 2.35, 'aqi': 3}","{'concentration': 16.86, 'aqi': 52}","{'concentration': 19.2, 'aqi': 17}",52
1,Miami,"{'concentration': 100.09, 'aqi': 1}","{'concentration': 0.27, 'aqi': 0}","{'concentration': 92.65, 'aqi': 144}","{'concentration': 0.73, 'aqi': 1}","{'concentration': 2.78, 'aqi': 9}","{'concentration': 7.34, 'aqi': 6}",144
2,Chicago,"{'concentration': 217.28, 'aqi': 2}","{'concentration': 10.59, 'aqi': 13}","{'concentration': 50.63, 'aqi': 42}","{'concentration': 3.1, 'aqi': 4}","{'concentration': 8.68, 'aqi': 28}","{'concentration': 10.64, 'aqi': 9}",42
3,Los Angeles,"{'concentration': 142.68, 'aqi': 1}","{'concentration': 6.76, 'aqi': 8}","{'concentration': 88.73, 'aqi': 134}","{'concentration': 2.22, 'aqi': 3}","{'concentration': 2.9, 'aqi': 9}","{'concentration': 8.8, 'aqi': 8}",134
4,Santa Monica,"{'concentration': 137.55, 'aqi': 1}","{'concentration': 5.12, 'aqi': 6}","{'concentration': 90.18, 'aqi': 137}","{'concentration': 1.77, 'aqi': 2}","{'concentration': 2.8, 'aqi': 9}","{'concentration': 9.04, 'aqi': 8}",137


In [10]:
aqi_columns = ["City", "CO" , "NO2", "O3", "SO2", "PM2.5", "PM10", "Air_Quality_Index"]
df_city.columns = aqi_columns
df_city.head()

,City,CO,NO2,O3,SO2,PM2.5,PM10,Air_Quality_Index
0,New York,"{'concentration': 370.18, 'aqi': 4}","{'concentration': 24.91, 'aqi': 31}","{'concentration': 32.75, 'aqi': 27}","{'concentration': 2.35, 'aqi': 3}","{'concentration': 16.86, 'aqi': 52}","{'concentration': 19.2, 'aqi': 17}",52
1,Miami,"{'concentration': 100.09, 'aqi': 1}","{'concentration': 0.27, 'aqi': 0}","{'concentration': 92.65, 'aqi': 144}","{'concentration': 0.73, 'aqi': 1}","{'concentration': 2.78, 'aqi': 9}","{'concentration': 7.34, 'aqi': 6}",144
2,Chicago,"{'concentration': 217.28, 'aqi': 2}","{'concentration': 10.59, 'aqi': 13}","{'concentration': 50.63, 'aqi': 42}","{'concentration': 3.1, 'aqi': 4}","{'concentration': 8.68, 'aqi': 28}","{'concentration': 10.64, 'aqi': 9}",42
3,Los Angeles,"{'concentration': 142.68, 'aqi': 1}","{'concentration': 6.76, 'aqi': 8}","{'concentration': 88.73, 'aqi': 134}","{'concentration': 2.22, 'aqi': 3}","{'concentration': 2.9, 'aqi': 9}","{'concentration': 8.8, 'aqi': 8}",134
4,Santa Monica,"{'concentration': 137.55, 'aqi': 1}","{'concentration': 5.12, 'aqi': 6}","{'concentration': 90.18, 'aqi': 137}","{'concentration': 1.77, 'aqi': 2}","{'concentration': 2.8, 'aqi': 9}","{'concentration': 9.04, 'aqi': 8}",137


#### Droping the irrelevant columns, not required for final analysis

In [11]:
# Dropping Local_Name and Country columns from public holiday dataframe
df_hd = df_holidays.drop(["Local_Name","Country"], axis =1)
df_hd.head()

,Holiday_Name,Date,Year,US_Regions,Is_Federal
0,New Year's Day,2025-01-01,2025,[],True
1,"Martin Luther King, Jr. Day",2025-01-20,2025,[],True
2,Presidents Day,2025-02-17,2025,[],True
3,Good Friday,2025-04-18,2025,"[US-CT, US-DE, US-HI, US-IN, US-KY, US-LA, US-...",False
4,Memorial Day,2025-05-26,2025,[],True


In [12]:
# Dropping CO, NO2, O3, SO2, PM2.5 and PM10 from Air Quality Index dataframe
df_ct = df_city.drop(["CO","NO2","O3","SO2","PM2.5","PM10"], axis =1)
df_ct.head(10)

,City,Air_Quality_Index
0,New York,52
1,Miami,144
2,Chicago,42
3,Los Angeles,134
4,Santa Monica,137
5,San Francisco,60


#### Converting "date" into datetime64 data type and remove "US-" from US_Regions

In [13]:
# Convert Date to datetime
df_hd["Date"] = pd.to_datetime(df_hd["Date"])
df_hd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Holiday_Name  13 non-null     object        
 1   Date          13 non-null     datetime64[ns]
 2   Year          13 non-null     int64         
 3   US_Regions    13 non-null     object        
 4   Is_Federal    13 non-null     bool          
dtypes: bool(1), datetime64[ns](1), int64(1), object(2)
memory usage: 561.0+ bytes


In [14]:
# Remove "US-" prefix from entries in US_Regions list
# Repalce [] with All Regions in US_Regions
df_hd["US_Regions"] = df_hd["US_Regions"].apply(
    lambda x: "All Regions" 
    if not x 
    else ", ".join(r.replace("US-", "") for r in x)
)
df_hd.head()

,Holiday_Name,Date,Year,US_Regions,Is_Federal
0,New Year's Day,2025-01-01,2025,All Regions,True
1,"Martin Luther King, Jr. Day",2025-01-20,2025,All Regions,True
2,Presidents Day,2025-02-17,2025,All Regions,True
3,Good Friday,2025-04-18,2025,"CT, DE, HI, IN, KY, LA, NC, ND, NJ, TN",False
4,Memorial Day,2025-05-26,2025,All Regions,True


#### Adding state column and value in AQI  dataframe

In [15]:
state_map = {
    "New York": "New York",
    "Miami": "Florida",
    "Chicago": "California",
    "Los Angeles": "California",
    "Santa Monica": "California",
    "San Francisco": "California"
}

# Map State
df_ct["State"] = df_ct["City"].map(state_map)

# Reorder columns
df_ct = df_ct[["City", "State", "Air_Quality_Index"]]
df_ct.head()

,City,State,Air_Quality_Index
0,New York,New York,52
1,Miami,Florida,144
2,Chicago,California,42
3,Los Angeles,California,134
4,Santa Monica,California,137


#### Finding any NaN and Outlier values

In [16]:
df_ct.head(10) # There is no NaN or Outlier values in Air Quality Index

,City,State,Air_Quality_Index
0,New York,New York,52
1,Miami,Florida,144
2,Chicago,California,42
3,Los Angeles,California,134
4,Santa Monica,California,137
5,San Francisco,California,60


In [17]:
df_ct.isnull().any() # If thre is any null values.

City                 False
State                False
Air_Quality_Index    False
dtype: bool

In [18]:
df_hd.head(10)

,Holiday_Name,Date,Year,US_Regions,Is_Federal
0,New Year's Day,2025-01-01,2025,All Regions,True
1,"Martin Luther King, Jr. Day",2025-01-20,2025,All Regions,True
2,Presidents Day,2025-02-17,2025,All Regions,True
3,Good Friday,2025-04-18,2025,"CT, DE, HI, IN, KY, LA, NC, ND, NJ, TN",False
4,Memorial Day,2025-05-26,2025,All Regions,True
5,Juneteenth National Independence Day,2025-06-19,2025,All Regions,True
6,Independence Day,2025-07-04,2025,All Regions,True
7,Labor Day,2025-09-01,2025,All Regions,True
8,Columbus Day,2025-10-13,2025,"AL, AZ, CO, CT, GA, ID, IL, IN, IA, KS, KY, LA...",False
9,Indigenous Peoples' Day,2025-10-13,2025,"AK, AL, CA, HI, IA, LA, ME, MI, MN, NC, NE, NM...",False


In [19]:
df_hd.isnull().any() # If thre is any null values.

Holiday_Name    False
Date            False
Year            False
US_Regions      False
Is_Federal      False
dtype: bool

In [21]:
# save dataframe to csv
df_hd.to_csv("df_us_holiday_api.csv")

In [22]:
# save dataframe to csv
df_ct.to_csv("df_us_aqi_api.csv")

### Ethical Considerations

The ethical considerations in fetching and handling the API data primarily involve getting relevant data for the academic work. API Ninjas allow us to use this data at no cost. Our API data is available for free with an account and an API Key.